In [1]:
#Rag Pipeline
from loaders import UniversalLoader
from embedders import OllamaEmbedder
from vectorstores import ChromaVectorStore
from ingesters import GlobIngester
import chunkers
import agents

from openai import OpenAI
import gradio as gr

c:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialization of Pipeline Class Instances for Ingestion

In [2]:
#Create Instances of Tools
embedder = OllamaEmbedder(model_name ="nomic-embed-text")
vectorstore = ChromaVectorStore(collection_name ="TemplateDB", embedder = embedder)
loader = UniversalLoader()
chunker = chunkers.FixedSizeChunkStrategy(max_tokens=800, overlap_tokens = 150)
ingester = GlobIngester(roots=["./TestDocuments"], doctype_filter={"pdf", "docx", "txt"})


### Document Ingestion

In [3]:
#Find files and load into database
files = ingester.collect()

for file in files:
    content = loader.load(file)
    chunks = chunker.chunk(content)
    content = [chunk.content for chunk in chunks]
    metadata = [chunk.metadata for chunk in chunks]
    vectorstore.add(content=content, metadata = metadata)
    print(f"Succefully added {len(chunks)} chunks for {file.name} to vector DB.")
        


Succefully added 1 chunks for TestDocument.txt to vector DB.


In [4]:
#Test Vector Store
vectorstore.query(query = "What's my dogs name?", top_k=1)['documents'][0]

['I am 29 years old\nI have dog named Ralph\nMy eyes are green\nI was born in May\nMy favorite color is forest green\nI am an engineer\nI have no hobbies\nMy name is sharko']

### Create Tools

In [5]:

information_lookup_desc = "Search the vector database for information if you do not have sufficient information to answer the question."
information_lookup_params = {
  "properties": {
      "query": {"type": "string", "description": "Natural language query to search."},
      "top_k": {"type": "integer", "description": "Number of results to return.", "default": 5, "maximum":50}
  },
  "required": ["query"]}

#function wrapper
information_lookup_function = lambda query, top_k=1: str(vectorstore.query(query, top_k)['documents'][0])

#Creation of tool object
information_lookup = agents.Function(function = information_lookup_function, name = "Information Lookup", description = information_lookup_desc, parameters = information_lookup_params, logging = True)



### Create Agents

In [9]:
ollama_url = "http://localhost:11434/v1"
client = OpenAI(api_key="ollama",base_url=ollama_url)

chatbot_model = "ministral-3:3b"
chatbot_prompt = """
   You are a helpful AI with access to user information.  Use the information lookup tool to answer personal questions related to the user"
 """
#create instance of chat agent
chatbot = agents.ChatAgent(system_prompt = chatbot_prompt, model=chatbot_model, client=client, tools=information_lookup)

#single call to the agent
chatbot.call("howdy!, what is my favorite color")

Information Lookup called with arguments: {'query': 'favorite color of the user who contacted you', 'top_k': 1}


'Your favorite color, according to the information I have, is **forest green**! Howdy again! 😊'

### Gradio UI

In [7]:
def chatwrapper(message,history):

    return chatbot.chat(message)
demo = gr.ChatInterface(fn=chatwrapper).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


error uploading: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


Information Lookup called with arguments: {'query': 'my name', 'top_k': 1}
